# Calculate Distortion Metrics for all Reconstructions
- Run Script: ~scripts/script_distortion_computation.py
- Resuts are saved in ~data/all_distortion_metrics

In [10]:
from pathlib import Path
import src.read_write as rw
import src.reconstruction as recon
import numpy as np

# ---- Directories ----
dir_reconstruction = Path('data/all_reconstructions')
dir_reconstruction.mkdir(parents=True, exist_ok=True)

dir_distortion_metrics = Path("data/all_distortion_metrics")
dir_distortion_metrics.mkdir(parents=True, exist_ok=True)

path_ground_truth = Path("data/ground_truth/phantom_MRI04_iso1mm_N4.nii.gz")
path_label_segmentations = Path("data/ground_truth/LABEL_SEGMENTATION_ALL.nii.gz")


In [ ]:
for file in dir_reconstruction.iterdir():
        if "position1" in file.name or "position2" in file.name or "position3" in file.name:
            print(f"Computing distortion metrics for {file.name}")
            !(python scripts/script_distortion_computation.py\
            -p {path_ground_truth}\
            -u {file}\
            -l {path_label_segmentations}\
            -o {dir_distortion_metrics})


Computing distortion metrics for position2_ROreg_ES000_IROC_Order12.nii.gz

Distortion Computation Pipeline (Phantom vs ULF)
Phantom:   data/ground_truth/phantom_MRI04_iso1mm_N4.nii.gz
ULF:       data/all_reconstructions/position2_ROreg_ES000_IROC_Order12.nii.gz
Output:    data/all_distortion_metrics/position2_ROreg_ES000_IROC_Order12

STEP 0: Copying original images
  ✓ Phantom copied → phantom.nii.gz
  ✓ ULF copied → ulf_mri.nii.gz
  ✓ Labels copied → labels.nii.gz

STEP 1: Resampling ULF to phantom resolution
  ✓ ULF resampled → ulf_mri_resampled.nii.gz
    Phantom spacing: (np.float32(1.0), np.float32(1.0), np.float32(1.0))

STEP 2: N4 bias field correction on resampled ULF
  ✓ N4 bias field correction applied → ulf_mri_n4.nii.gz

STEP 3: Thresholding images
  Processing phantom: phantom_MRI04_iso1mm_N4
    → phantom_threshold-otsu.nii.gz (threshold=2584.7)
  Processing ULF: position2_ROreg_ES000_IROC_Order12
    → ulf_threshold-otsu.nii.gz (threshold=0.7)

STEP 4: ANTs registratio

# Calculate Distortion Metrics Averaged accross Positions
- Run Script: ~scripts/script_distortion_computation_position_avg.py
- Resuts are saved as Position 0 in ~data/all_distortion_metrics

In [12]:
from pathlib import Path


for dir_pos1 in dir_distortion_metrics.iterdir():
    if "position1" in dir_pos1.name:
        #---- Match Files to be processed in List ----
        dir_pos2 = dir_distortion_metrics / dir_pos1.name.replace("position1", "position2")
        dir_pos3 = dir_distortion_metrics / dir_pos1.name.replace("position1", "position3")

        file_pos1 = dir_pos1 / "phantom_2_ulf_distortion-magnitude.nii.gz"
        file_pos2 = dir_pos2 / "phantom_2_ulf_distortion-magnitude.nii.gz"
        file_pos3 = dir_pos3 / "phantom_2_ulf_distortion-magnitude.nii.gz"

        file_pos1_by_label = dir_pos1 / "phantom_2_ulf_distortion-magnitude-by-label.nii.gz"
        file_pos2_by_label = dir_pos2 / "phantom_2_ulf_distortion-magnitude-by-label.nii.gz"
        file_pos3_by_label = dir_pos3 / "phantom_2_ulf_distortion-magnitude-by-label.nii.gz"

        # ---- Define Final Export directory ----
        path_export_dir_full = dir_distortion_metrics / dir_pos1.name.replace("position1", "position0")

        print(path_export_dir_full)

        !(python scripts/script_distortion_computation_position_avg.py \
        -u \
        {file_pos1} \
        {file_pos2} \
        {file_pos3} \
        -b \
        {file_pos1_by_label} \
        {file_pos2_by_label} \
        {file_pos3_by_label} \
        -l {path_label_segmentations} \
        -o {path_export_dir_full})


data/all_distortion_metrics/position0_ROreg_ES000_CPROne_Order04
/Users/robin/Documents/_2_Code/IROC-Paper-v2/scripts/script_distortion_computation_position_avg.py:32: SyntaxWarning: invalid escape sequence '\ '
  data/all_distortion_metrics/position1.../phantom_2_ulf_distortion-magnitude.nii.gz \           # Calculated by using "script_distortion_computation.py"
STEP 1: Averaging distortion magnitude maps
  Loading data/all_distortion_metrics/position1_ROreg_ES000_CPROne_Order04/phantom_2_ulf_distortion-magnitude.nii.gz
  Loading data/all_distortion_metrics/position2_ROreg_ES000_CPROne_Order04/phantom_2_ulf_distortion-magnitude.nii.gz
  Loading data/all_distortion_metrics/position3_ROreg_ES000_CPROne_Order04/phantom_2_ulf_distortion-magnitude.nii.gz
  ✓ Averaged 3 distortion maps

  ✓ Distortion magnitude → phantom_2_ulf_distortion-magnitude.nii.gz

STEP 3: Computing distortion magnitude per label
  ✓ Distortion magnitude by label → phantom_2_ulf_distortion-magnitude-per-label.nii.gz 